# Database Visualization Dashboard

**Purpose:** Interactive visualizations for presenting database content and analysis results

**Tech Stack:** Plotly for interactive charts

**Structure:**
1. Individual Analysis Run Exploration (clustering & classification)
2. Model Performance Comparison
3. Label Performance Comparison  
4. Model Performance Across Runs (pooled)

---

## Setup

In [ ]:
# Install plotly if needed (run once)
# !pip install plotly

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import psycopg2
from dotenv import load_dotenv
import os

# Load environment variables
# load_dotenv('.env')
load_dotenv('.env.test')

# Database connection
conn = psycopg2.connect(
    host=os.getenv('DB_HOST'),
    port=os.getenv('DB_PORT'),
    user=os.getenv('DB_USER'),
    password=os.getenv('DB_PASSWORD', ''),
    # database='image_analysis_dev'
    database='image_analysis_test'
)

print("✅ Connected to database")

In [ ]:
import plotly.io as pio
pio.renderers.default = "iframe"  # or try "notebook" or "jupyterlab"

In [ ]:
import sys
print(sys.executable)

---
# 1. Individual Analysis Run Exploration

Examine specific analysis runs in detail - both metadata and results.

---

## 1.1 Clustering Analysis

### Clustering Metadata
---
#### Including analyses with pretrained model (no training and validation data).
---

In [ ]:
# Query 1: Clustering metadata (YOUR CODE) 
# Shows: autoencoder, dim reduction, clustering method

clustering_metadata_query = """
-------------------------------------------------------------------------------------
-- Present clustering metadata:
-------------------------------------------------------------------------------------
with analysed_images_clustering as
(
	select 
	ar.analysis_run_id, 
	count(clustering_id) as number_images_analysed
	
	from clustering_results cr
		join analysis_runs ar
			on cr.analysis_run_id = ar.analysis_run_id
	where ar.analysis_type = 'clustering'
	group by ar.analysis_run_id
)

select
	ar.analysis_run_id, 
    ar.model_name,
	autoencoder_name,
    python_script,
	dim_reduction_name,
	dim_reduction_params,
	clustering_name,
	aic.number_images_analysed
	
from analysis_runs ar
	join analysed_images_clustering aic
		on ar.analysis_run_id = aic.analysis_run_id
where ar.analysis_type = 'clustering'
order by analysis_run_id
"""

clustering_meta = pd.read_sql(clustering_metadata_query, conn)
display(clustering_meta)

### Clustering Metadata
---
#### Only analysis-runs with training from scratch, including training and validation data.
---

In [ ]:
# Query 1: Clustering metadata (YOUR CODE)
# Shows: autoencoder, dim reduction, clustering method, train/val/test counts

clustering_metadata_query = """
-------------------------------------------------------------------------------------
-- Present clustering metadata:
-------------------------------------------------------------------------------------
with analysed_images_clustering as
(
	select 
	ar.analysis_run_id, 
	count(clustering_id) as number_images_analysed
	
	from clustering_results cr
		join analysis_runs ar
			on cr.analysis_run_id = ar.analysis_run_id
	where ar.analysis_type = 'clustering'
	group by ar.analysis_run_id
),

train_val_images as
(
	select 
		ar.analysis_run_id,
		sum(case when tvs.split_type='train' then 1 else 0 end) as number_training_images,
		sum(case when tvs.split_type='validation' then 1 else 0 end) as number_validation_images
	
	from analysis_runs ar
		join training_validation_splits tvs
			on ar.analysis_run_id = tvs.analysis_run_id
	where ar.analysis_type = 'clustering'
	group by
		ar.analysis_run_id
)

select
	ar.analysis_run_id, 
    ar.model_name,
	autoencoder_name,
    python_script,
	dim_reduction_name,
	dim_reduction_params,
	clustering_name,
	tvi.number_training_images,
	tvi.number_validation_images,
	aic.number_images_analysed
	
from analysis_runs ar
	join analysed_images_clustering aic
		on ar.analysis_run_id = aic.analysis_run_id
	join train_val_images tvi
		on ar.analysis_run_id = tvi.analysis_run_id
where ar.analysis_type = 'clustering'
order by analysis_run_id
"""

clustering_meta = pd.read_sql(clustering_metadata_query, conn)
display(clustering_meta)

In [ ]:
# Visualization: Train/Val/Test split breakdown
# Change analysis_run_id here to explore different runs
TARGET_RUN_ID = 9  # TODO: Change this to explore different clustering runs

run_data = clustering_meta[clustering_meta['analysis_run_id'] == TARGET_RUN_ID]

if len(run_data) > 0:
    # Create data for pie chart
    split_data = pd.DataFrame({
        #'Split Type': ['Training', 'Validation', 'Test'],
        'Split Type': ['Training', 'Validation'],
        'Count': [
            run_data['number_training_images'].values[0],
            run_data['number_validation_images'].values[0]
            #run_data['number_images_analysed'].values[0]
        ]
    })
    
    fig = px.pie(
        split_data, 
        values='Count', 
        names='Split Type',
        title=f'Data Split - Analysis Run {TARGET_RUN_ID}<br><sub>{run_data["autoencoder_name"].values[0]} → {run_data["dim_reduction_name"].values[0]} → {run_data["clustering_name"].values[0]}</sub>',
        color_discrete_sequence=px.colors.qualitative.Set2
    )
    fig.update_traces(textposition='inside', textinfo='percent+label+value')
    fig.show()
else:
    print(f"No clustering data found for analysis_run_id={TARGET_RUN_ID}")

In [ ]:
# Query 2: Clustering metadata (YOUR CODE)
# Shows: autoencoder, dim reduction, clustering method, label distribution

clustering_metadata_query = """
-------------------------------------------------------------------------------------
-- Present clustering metadata:
-------------------------------------------------------------------------------------
with analysed_images_clustering as
(
	select 
	ar.analysis_run_id, 
	count(clustering_id) as number_images_analysed
	
	from clustering_results cr
		join analysis_runs ar
			on cr.analysis_run_id = ar.analysis_run_id
	where ar.analysis_type = 'clustering'
	group by ar.analysis_run_id
),

train_val_images as
(
	select 
		ar.analysis_run_id,
        tvs.split_type,
		sum(case when gtl.value='true' then 1 else 0 end) as number_is_map,
		sum(case when gtl.value='false' then 1 else 0 end) as number_not_map
	
	from analysis_runs ar
		join training_validation_splits tvs
			on ar.analysis_run_id = tvs.analysis_run_id
                join ground_truth_labels gtl
                    on tvs.image_id = gtl.image_id
	where ar.analysis_type = 'clustering'
    and gtl.label_name = 'is_map'
	group by
		ar.analysis_run_id,
        tvs.split_type
)

select
	ar.analysis_run_id, 
    ar.model_name,
	autoencoder_name,
    python_script,
	dim_reduction_name,
	dim_reduction_params,
	clustering_name,
    tvi.split_type,
	tvi.number_is_map,
	tvi.number_not_map
	
from analysis_runs ar
	join analysed_images_clustering aic
		on ar.analysis_run_id = aic.analysis_run_id
	join train_val_images tvi
		on ar.analysis_run_id = tvi.analysis_run_id
where ar.analysis_type = 'clustering'
order by analysis_run_id, tvi.split_type
"""

clustering_meta_2 = pd.read_sql(clustering_metadata_query, conn)
display(clustering_meta_2)

In [ ]:
# Visualization: Train/Val/Test split breakdown
# Change analysis_run_id here to explore different runs
TARGET_RUN_ID = 7  # TODO: Change this to explore different clustering runs

run_data = clustering_meta_2[clustering_meta_2['analysis_run_id'] == TARGET_RUN_ID]

if len(run_data) > 0:
    # Create data for pie chart
    split_data = pd.DataFrame({
        #'Split Type': ['Training', 'Validation', 'Test'],
        'Split Type': ['Training', 'Validation'],
        'Count': [
            run_data['number_is_map'].values[0],
            run_data['number_not_map'].values[0]
            #run_data['number_images_analysed'].values[0]
        ]
    })
    
    fig = px.pie(
        split_data, 
        values='Count', 
        names='Split Type',
        title=f'Data Split - Analysis Run {TARGET_RUN_ID}<br><sub>{run_data["autoencoder_name"].values[0]} → {run_data["dim_reduction_name"].values[0]} → {run_data["clustering_name"].values[0]}</sub>',
        color_discrete_sequence=px.colors.qualitative.Set2
    )
    fig.update_traces(textposition='inside', textinfo='percent+label+value')
    fig.show()
else:
    print(f"No clustering data found for analysis_run_id={TARGET_RUN_ID}")

### Clustering Results

In [ ]:
# Query 2: Clustering results - cluster quality (YOUR CODE)
# Shows: How well clusters separate 'is_map' label

clustering_results_query = """
-------------------------------------------------------------------------------------
-- Present clustering results:
-------------------------------------------------------------------------------------
select 
	ar.analysis_run_id,
	label_name,
    python_script,
	cr.cluster_id,
	count(cr.clustering_id) as number_images_in_cluster,
	round(sum(case when value = 'true' then 1 else 0 end)::numeric/count(value), 2) as proportion_of_maps
		
from analysis_runs ar
	 join clustering_results cr
	 	on ar.analysis_run_id = cr.analysis_run_id
			join ground_truth_labels gtl
				on cr.image_id = gtl.image_id
where ar.analysis_type = 'clustering'
	and gtl.label_name = 'is_map'
group by
	ar.analysis_run_id,
	cr.cluster_id,
	label_name
"""

clustering_results = pd.read_sql(clustering_results_query, conn)
display(clustering_results)

In [ ]:
# Visualization: Cluster composition (proportion of maps per cluster)
# Change TARGET_RUN_ID to explore different clustering runs
TARGET_RUN_ID = 11

run_results = clustering_results[clustering_results['analysis_run_id'] == TARGET_RUN_ID]

if len(run_results) > 0:
    fig = go.Figure()
    
    # Add bar for proportion of maps
    fig.add_trace(go.Bar(
        x=run_results['cluster_id'],
        y=run_results['proportion_of_maps'],
        name='Proportion Maps',
        marker_color='indianred',
        text=run_results['proportion_of_maps'],
        texttemplate='%{text:.0%}',
        textposition='outside'
    ))
    
    # Add cluster size as secondary trace
    fig.add_trace(go.Scatter(
        x=run_results['cluster_id'],
        y=run_results['number_images_in_cluster'],
        name='Cluster Size',
        yaxis='y2',
        mode='markers+lines',
        marker=dict(size=10, color='lightseagreen'),
        line=dict(width=2)
    ))
    
    fig.update_layout(
        title=f'Cluster Quality - Analysis Run {TARGET_RUN_ID}<br><sub>Map Concentration vs Cluster Size</sub>',
        xaxis_title='Cluster ID',
        yaxis_title='Proportion of Maps',
        yaxis2=dict(
            title='Number of Images',
            overlaying='y',
            side='right'
        ),
        hovermode='x unified',
        height=500
    )
    
    fig.show()
else:
    print(f"No clustering results found for analysis_run_id={TARGET_RUN_ID}")

In [ ]:
# Cell A: Explore available clusters for an analysis run

ANALYSIS_RUN_ID = 8  # TODO: Change this to your analysis run

# Query to see cluster composition
cluster_info_query = f"""
SELECT 
    cluster_id,
    COUNT(*) as image_count,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) as percentage
FROM clustering_results
WHERE analysis_run_id = {ANALYSIS_RUN_ID}
GROUP BY cluster_id
ORDER BY cluster_id
"""

cluster_info = pd.read_sql(cluster_info_query, conn)
print(f"Analysis Run {ANALYSIS_RUN_ID} - Cluster Summary:")
display(cluster_info)
print(f"\nTotal clusters: {len(cluster_info)}")
print(f"Total images: {cluster_info['image_count'].sum()}")

In [ ]:
# Cell B Alternative: Display images from selected clusters (GRID LAYOUT)

from PIL import Image
from IPython.display import display as ipython_display, HTML
import base64
from io import BytesIO

# ============================================================================
# PARAMETERS - Change these
# ============================================================================
ANALYSIS_RUN_ID = 9  # Which analysis run
CLUSTERS_TO_SHOW = [0, 1, 2, 3]  # Which clusters (list)
IMAGES_PER_CLUSTER = 50  # How many images to show per cluster
IMAGES_PER_ROW = 3  # Grid layout: images per row
MAX_IMAGE_WIDTH = 300  # Resize images to this width (pixels)

# ============================================================================
# Query images from selected clusters
# ============================================================================

clusters_str = ','.join(map(str, CLUSTERS_TO_SHOW))

cluster_images_query = f"""
WITH ranked_images AS (
    SELECT 
        cr.cluster_id,
        cr.image_id,
        i.filename,
        i.file_path,
        ROW_NUMBER() OVER (PARTITION BY cr.cluster_id ORDER BY cr.clustering_id) as rn
    FROM clustering_results cr
    JOIN images i ON cr.image_id = i.image_id
    WHERE cr.analysis_run_id = {ANALYSIS_RUN_ID}
        AND cr.cluster_id IN ({clusters_str})
)
SELECT cluster_id, image_id, filename, file_path
FROM ranked_images
WHERE rn <= {IMAGES_PER_CLUSTER}
ORDER BY cluster_id, rn
"""

cluster_images = pd.read_sql(cluster_images_query, conn)

if len(cluster_images) == 0:
    print(f"No images found for analysis_run_id={ANALYSIS_RUN_ID}, clusters={CLUSTERS_TO_SHOW}")
else:
    print(f"Displaying {len(cluster_images)} images across {len(CLUSTERS_TO_SHOW)} clusters\n")
    
    # Group by cluster
    for cluster_id in CLUSTERS_TO_SHOW:
        cluster_imgs = cluster_images[cluster_images['cluster_id'] == cluster_id]
        
        if len(cluster_imgs) == 0:
            continue
            
        print("=" * 80)
        print(f"CLUSTER {cluster_id} ({len(cluster_imgs)} images)")
        print("=" * 80)
        
        # Build HTML grid for this cluster
        html_parts = [f'<div style="display: grid; grid-template-columns: repeat({IMAGES_PER_ROW}, 1fr); gap: 10px; margin: 20px 0;">']
        
        for idx, (_, row) in enumerate(cluster_imgs.iterrows()):
            try:
                img = Image.open(row['file_path'])
                
                # Resize for display
                aspect_ratio = img.height / img.width
                new_height = int(MAX_IMAGE_WIDTH * aspect_ratio)
                img_resized = img.resize((MAX_IMAGE_WIDTH, new_height))
                
                # Convert to base64 for HTML embedding
                buffered = BytesIO()
                img_resized.save(buffered, format="PNG")
                img_str = base64.b64encode(buffered.getvalue()).decode()
                
                # Add image to grid
                html_parts.append(f'''
                <div style="text-align: center;">
                    <img src="data:image/png;base64,{img_str}" style="max-width: 100%; height: auto;">
                    <div style="font-size: 12px; margin-top: 5px;">{row['filename']}<br>(ID: {row['image_id']})</div>
                </div>
                ''')
                
            except Exception as e:
                print(f"Could not load: {row['file_path']} - {e}")
        
        html_parts.append('</div>')
        
        # Display the grid
        ipython_display(HTML(''.join(html_parts)))
        print("\n")

## 1.2 Classification Analysis (Pretrained Models)

### Classification Metadata

In [ ]:
# Query 3: Classification metadata (YOUR CODE)

# Get run metadata
classification_meta_query = f"""
select
	analysis_run_id,
	analysis_type,
	model_name,
	python_script,
	model_version,
	notes,
	images_processed,
    duration_seconds

from analysis_runs;
"""

classification_meta = pd.read_sql(classification_meta_query, conn)
print("\n📋 Classification Run Metadata:")
display(classification_meta)

# Get labels predicted
labels_predicted_query = f"""
select
	distinct pr.label_name

from analysis_runs ar
	join predictions pr
		on ar.analysis_run_id = pr.analysis_run_id
"""

labels_predicted = pd.read_sql(labels_predicted_query, conn)
print("\n🏷️  Labels Predicted:")
display(labels_predicted)

In [ ]:
# Query 3: Classification metadata (YOUR CODE)
# Change the analysis_run_id parameter to explore different classification runs

CLASSIFICATION_RUN_ID = 6  # TODO: Change this to explore different classification runs

# Get run metadata
classification_meta_query = f"""
select
	analysis_run_id,
	analysis_type,
	model_name,
	python_script,
	model_version,
	notes,
	images_processed

from analysis_runs
where analysis_run_id={CLASSIFICATION_RUN_ID}
"""

classification_meta = pd.read_sql(classification_meta_query, conn)
print("\n📋 Classification Run Metadata:")
display(classification_meta)

# Get labels predicted
labels_predicted_query = f"""
select
	distinct pr.label_name

from analysis_runs ar
	join predictions pr
		on ar.analysis_run_id = pr.analysis_run_id
where ar.analysis_run_id={CLASSIFICATION_RUN_ID}
"""

labels_predicted = pd.read_sql(labels_predicted_query, conn)
print("\n🏷️  Labels Predicted:")
display(labels_predicted)

### Classification Results & Performance

In [ ]:
# Query 4: Classification performance by label (YOUR CODE)
# Change CLASSIFICATION_RUN_ID to match the run you want to analyze
CLASSIFICATION_RUN_ID = 6

classification_performance_query = f"""
with performance_analysis_run as
(
	select
		ar.analysis_run_id,
		ar.analysis_type,
		ar.model_name,
		ar.python_script,
		pr.image_id,
		pr.label_name,
		gtl.value,
		pr.predicted_value,
		case when gtl.value = pr.predicted_value then 1 else 0 end as match
	
	from analysis_runs ar
		join predictions pr
			on ar.analysis_run_id = pr.analysis_run_id
		join ground_truth_labels gtl
			on pr.image_id = gtl.image_id and pr.label_name = gtl.label_name
	where ar.analysis_run_id={CLASSIFICATION_RUN_ID}
	order by image_id, gtl.label_name
)

select 
analysis_run_id,
model_name,
label_name, 
round(sum(match)::numeric/count(*), 2) as proportion_correct_answers
from performance_analysis_run
group by analysis_run_id, model_name, label_name
"""

classification_perf = pd.read_sql(classification_performance_query, conn)
display(classification_perf)

In [ ]:
# Visualization: Accuracy by label for this classification run

if len(classification_perf) > 0:
    fig = px.bar(
        classification_perf,
        x='label_name',
        y='proportion_correct_answers',
        title=f'Classification Accuracy by Label - Run {CLASSIFICATION_RUN_ID}<br><sub>Model: {classification_perf["model_name"].values[0]}</sub>',
        labels={'proportion_correct_answers': 'Accuracy', 'label_name': 'Label'},
        text='proportion_correct_answers',
        color='proportion_correct_answers',
        color_continuous_scale='RdYlGn',
        range_color=[0, 1]
    )
    
    fig.update_traces(texttemplate='%{text:.0%}', textposition='outside')
    fig.update_yaxes(range=[0, 1.1], tickformat='.0%')  # Note: yaxes (plural)
    fig.update_layout(height=500, showlegend=False)
    fig.show()
else:
    print(f"No classification performance data found for analysis_run_id={CLASSIFICATION_RUN_ID}")

---
# 2. Model Performance Comparison

Compare accuracy across different models using the `model_performance` view.

---

In [ ]:
# Use model_performance view to compare models
model_comparison_query = """
SELECT 
    model_name,
    label_name,
    accuracy_percentage,
    total_predictions,
    avg_confidence
FROM model_performance
ORDER BY model_name, label_name
"""

model_comparison = pd.read_sql(model_comparison_query, conn)
display(model_comparison)

In [ ]:
# Visualization: Model comparison - accuracy heatmap

# Pivot data for heatmap
heatmap_data = model_comparison.pivot(
    index='label_name', 
    columns='model_name', 
    values='accuracy_percentage'
)

# fig = go.Figure(data=go.Heatmap(
#     z=heatmap_data.values,
#     x=heatmap_data.columns,
#     y=heatmap_data.index,
#     #colorscale='RdYlGn',
#     colorscale='Plasma',
#     #colorscale='Blues',
#     #colorscale='Viridis',
#     text=heatmap_data.values,
#     texttemplate='%{text:.1f}%',
#     textfont={"size": 12},
#     colorbar=dict(title="Accuracy %")
# ))

fig = go.Figure(data=go.Heatmap(
    z=heatmap_data.values,
    x=heatmap_data.columns,
    y=heatmap_data.index,
    colorscale='RdYlGn',
    text=heatmap_data.values,
    texttemplate='%{text:.1f}%',
    textfont={"size": 12, "color": "black"},  # Changed to black for better contrast
    colorbar=dict(title="Accuracy %")
))

fig.update_layout(
    title='Model Performance Comparison<br><sub>Accuracy by Model and Label</sub>',
    xaxis_title='Model',
    yaxis_title='Label',
    height=500
)

fig.show()

In [ ]:
# Visualization: Model comparison - grouped bar chart

fig = px.bar(
    model_comparison,
    x='label_name',
    y='accuracy_percentage',
    color='model_name',
    barmode='group',
    title='Model Performance Comparison<br><sub>Accuracy by Label - All Models</sub>',
    labels={'accuracy_percentage': 'Accuracy (%)', 'label_name': 'Label'},
    text='accuracy_percentage'
)

fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig.update_layout(height=500, yaxis_range=[0, 110])
fig.show()

---
# 3. Label Performance Comparison

Compare how well different labels are predicted across all models.

---

In [ ]:
# Aggregate performance by label (across all models)
label_performance_query = """
SELECT 
    label_name,
    COUNT(DISTINCT model_name) as models_tested,
    AVG(accuracy_percentage) as avg_accuracy,
    MIN(accuracy_percentage) as min_accuracy,
    MAX(accuracy_percentage) as max_accuracy,
    SUM(total_predictions) as total_predictions
FROM model_performance
GROUP BY label_name
ORDER BY avg_accuracy DESC
"""

label_performance = pd.read_sql(label_performance_query, conn)
display(label_performance)

In [ ]:
# Visualization: Label difficulty ranking

fig = go.Figure()

# Add bars for average accuracy
fig.add_trace(go.Bar(
    x=label_performance['label_name'],
    y=label_performance['avg_accuracy'],
    name='Average Accuracy',
    marker_color='steelblue',
    text=label_performance['avg_accuracy'],
    texttemplate='%{text:.1f}%',
    textposition='outside'
))

# Add error bars for min/max range
fig.add_trace(go.Scatter(
    x=label_performance['label_name'],
    y=label_performance['max_accuracy'],
    mode='markers',
    name='Max Accuracy',
    marker=dict(size=8, color='green', symbol='triangle-up'),
    showlegend=True
))

fig.add_trace(go.Scatter(
    x=label_performance['label_name'],
    y=label_performance['min_accuracy'],
    mode='markers',
    name='Min Accuracy',
    marker=dict(size=8, color='red', symbol='triangle-down'),
    showlegend=True
))

fig.update_layout(
    title='Label Prediction Difficulty<br><sub>Average Accuracy Across All Models (with Min/Max Range)</sub>',
    xaxis_title='Label',
    yaxis_title='Accuracy (%)',
    height=500,
    yaxis_range=[0, 110],
    hovermode='x unified'
)

fig.show()

---
# 4. Model Performance Across Runs (Pooled)

Aggregate performance by model across all analysis runs.

---

In [ ]:
# Performance pooled across all runs for each model
# Adapted from model_performance view - aggregates across runs

model_pooled_query = """
SELECT 
    ar.model_name,
    p.label_name,
    COUNT(DISTINCT ar.analysis_run_id) as number_of_runs,
    COUNT(*) as total_predictions,
    SUM(CASE WHEN p.predicted_value = gt.value THEN 1 ELSE 0 END) as correct_predictions,
    ROUND(
        100.0 * SUM(CASE WHEN p.predicted_value = gt.value THEN 1 ELSE 0 END) / COUNT(*),
        2
    ) as accuracy_percentage,
    AVG(p.confidence_score) as avg_confidence
FROM predictions p
JOIN analysis_runs ar ON p.analysis_run_id = ar.analysis_run_id
JOIN ground_truth_history gt 
    ON p.image_id = gt.image_id 
    AND p.label_name = gt.label_name 
    AND gt.is_current = TRUE
GROUP BY ar.model_name, p.label_name
ORDER BY ar.model_name, p.label_name
"""

model_pooled = pd.read_sql(model_pooled_query, conn)
display(model_pooled)

In [ ]:
# Visualization: Overall model performance (pooled across all runs)

# Calculate overall accuracy per model (average across labels)
model_overall = model_pooled.groupby('model_name').agg({
    'total_predictions': 'sum',
    'correct_predictions': 'sum',
    'number_of_runs': 'mean'
}).reset_index()

model_overall['overall_accuracy'] = (
    model_overall['correct_predictions'] / model_overall['total_predictions'] * 100
)

fig = px.bar(
    model_overall,
    x='model_name',
    y='overall_accuracy',
    title='Overall Model Performance<br><sub>Pooled Across All Analysis Runs</sub>',
    labels={'overall_accuracy': 'Accuracy (%)', 'model_name': 'Model'},
    text='overall_accuracy',
    color='overall_accuracy',
    color_continuous_scale='RdYlGn',
    range_color=[0, 100]
)

fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig.update_layout(height=500, yaxis_range=[0, 110], showlegend=False)
fig.show()

In [ ]:
# Model performance + duration across runs (pooled)

model_pooled_with_duration_query = """
SELECT 
    ar.model_name,
    COUNT(DISTINCT ar.analysis_run_id) as number_of_runs,
    SUM(ar.duration_seconds) as total_duration_seconds,
    ROUND(AVG(ar.duration_seconds)::NUMERIC, 1) as avg_duration_seconds,
    COUNT(DISTINCT p.image_id) as total_images,
    COUNT(*) as total_predictions,
    SUM(CASE WHEN p.predicted_value = gt.value THEN 1 ELSE 0 END) as correct_predictions,
    ROUND(
        (100.0 * SUM(CASE WHEN p.predicted_value = gt.value THEN 1 ELSE 0 END) / COUNT(*))::NUMERIC,
        2
    ) as accuracy_percentage
FROM analysis_runs ar
JOIN predictions p ON ar.analysis_run_id = p.analysis_run_id
JOIN ground_truth_history gt 
    ON p.image_id = gt.image_id 
    AND p.label_name = gt.label_name 
    AND gt.is_current = TRUE
GROUP BY ar.model_name
ORDER BY accuracy_percentage DESC
"""

model_pooled_duration = pd.read_sql(model_pooled_with_duration_query, conn)

# Add formatted duration columns for readability
model_pooled_duration['avg_duration_formatted'] = model_pooled_duration['avg_duration_seconds'].apply(
    lambda x: f"{int(x//60)}m {int(x%60)}s" if x >= 60 else f"{x:.1f}s"
)

display(model_pooled_duration)

In [ ]:
# Visualization: Model efficiency (accuracy vs speed) - IMPROVED

fig = go.Figure()

# Calculate bubble sizes with better scaling
min_size = 10  # Minimum bubble size
max_size = 50  # Maximum bubble size
predictions = model_pooled_duration['total_predictions']
size_scale = (predictions - predictions.min()) / (predictions.max() - predictions.min()) if len(predictions) > 1 else 0.5
bubble_sizes = min_size + (size_scale * (max_size - min_size))

# Scatter plot: accuracy vs average duration
fig.add_trace(go.Scatter(
    x=model_pooled_duration['avg_duration_seconds'],
    y=model_pooled_duration['accuracy_percentage'],
    mode='markers+text',
    marker=dict(
        size=bubble_sizes,
        color=model_pooled_duration['accuracy_percentage'],
        colorscale='RdYlGn',
        showscale=True,
        colorbar=dict(title="Accuracy %"),
        line=dict(width=2, color='black')  # Add border for visibility
    ),
    text=model_pooled_duration['model_name'],
    textposition='top center',
    textfont=dict(size=12),
    hovertemplate='<b>%{text}</b><br>Accuracy: %{y:.1f}%<br>Avg Duration: %{x:.1f}s<br>Predictions: ' + 
                  model_pooled_duration['total_predictions'].astype(str) + '<extra></extra>'
))

fig.update_layout(
    title='Model Efficiency: Accuracy vs Speed<br><sub>Bubble size = total predictions</sub>',
    xaxis_title='Average Duration per Run (seconds)',
    yaxis_title='Accuracy (%)',
    height=500,
    yaxis_range=[0, 110]
)

fig.show()

In [ ]:
# See what data we have
display(model_pooled_duration[['model_name', 'avg_duration_seconds', 'accuracy_percentage', 'total_predictions']])

In [ ]:
# Model performance + duration across runs (pooled)

model_pooled_with_duration_query = """
SELECT 
    ar.model_name,
    CASE 
        WHEN ar.notes LIKE '%Loaded: Once in workflow.%' THEN 'Loaded Once'
        ELSE 'Loaded every iteration'
    END as notes_category,
    COUNT(DISTINCT ar.analysis_run_id) as number_of_runs,
    SUM(ar.duration_seconds) as total_duration_seconds,
    ROUND(AVG(ar.duration_seconds)::NUMERIC, 1) as avg_duration_seconds,
    COUNT(DISTINCT p.image_id) as total_images,
    COUNT(*) as total_predictions,
    SUM(CASE WHEN p.predicted_value = gt.value THEN 1 ELSE 0 END) as correct_predictions,
    ROUND(
        (100.0 * SUM(CASE WHEN p.predicted_value = gt.value THEN 1 ELSE 0 END) / COUNT(*))::NUMERIC,
        2
    ) as accuracy_percentage
FROM analysis_runs ar
JOIN predictions p ON ar.analysis_run_id = p.analysis_run_id
JOIN ground_truth_history gt 
    ON p.image_id = gt.image_id 
    AND p.label_name = gt.label_name 
    AND gt.is_current = TRUE
GROUP BY ar.model_name, 
    CASE 
        WHEN ar.notes LIKE '%Loaded: Once in workflow.%' THEN 'Loaded Once'
        ELSE 'Loaded every iteration'
    END  -- Must repeat the CASE expression
ORDER BY ar.model_name, accuracy_percentage DESC
"""

model_pooled_duration = pd.read_sql(model_pooled_with_duration_query, conn)

# Add formatted duration columns for readability
model_pooled_duration['avg_duration_formatted'] = model_pooled_duration['avg_duration_seconds'].apply(
    lambda x: f"{int(x//60)}m {int(x%60)}s" if x >= 60 else f"{x:.1f}s"
)

display(model_pooled_duration)

In [ ]:
# Visualization: Model efficiency (accuracy vs speed) - No overlapping labels

fig = go.Figure()

# Calculate bubble sizes with better scaling
min_size = 10  # Minimum bubble size
max_size = 50  # Maximum bubble size
predictions = model_pooled_duration['total_predictions']
size_scale = (predictions - predictions.min()) / (predictions.max() - predictions.min()) if len(predictions) > 1 else 0.5
bubble_sizes = min_size + (size_scale * (max_size - min_size))

# Map categories to symbols
symbols = model_pooled_duration['notes_category'].map({
    'Loaded Once': 'circle',
    'Loaded every iteration': 'square'
})

# Scatter plot: accuracy vs average duration
fig.add_trace(go.Scatter(
    x=model_pooled_duration['avg_duration_seconds'],
    y=model_pooled_duration['accuracy_percentage'],
    mode='markers',  # Removed 'text' - only markers now
    marker=dict(
        size=bubble_sizes,
        symbol=symbols,  # Different shapes for different categories
        color=model_pooled_duration['accuracy_percentage'],
        colorscale='RdYlGn',
        showscale=True,
        colorbar=dict(title="Accuracy %"),
        line=dict(width=2, color='black')  # Add border for visibility
    ),
    hovertemplate=(
        '<b>%{customdata[0]}</b><br>' +
        'Category: %{customdata[1]}<br>' +
        'Accuracy: %{y:.1f}%<br>' +
        'Avg Duration: %{x:.1f}s<br>' +
        'Predictions: %{customdata[2]}<extra></extra>'
    ),
    customdata=model_pooled_duration[['model_name', 'notes_category', 'total_predictions']]
))

fig.update_layout(
    title='Model Efficiency: Accuracy vs Speed<br><sub>Bubble size = total predictions | Circle = Loaded Once, Square = Loaded every iteration | Hover for details</sub>',
    xaxis_title='Average Duration per Run (seconds)',
    yaxis_title='Accuracy (%)',
    height=500,
    yaxis_range=[0, 110]
)

fig.show()

In [ ]:
# See what data we have
display(model_pooled_duration[['model_name', 'notes_category', 'avg_duration_seconds', 'accuracy_percentage', 'total_predictions']])

In [ ]:
# Visualization: Model performance by label (pooled)

fig = px.bar(
    model_pooled,
    x='model_name',
    y='accuracy_percentage',
    color='label_name',
    barmode='group',
    title='Model Performance by Label<br><sub>Pooled Across All Runs</sub>',
    labels={'accuracy_percentage': 'Accuracy (%)', 'model_name': 'Model'},
    text='accuracy_percentage'
)

fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig.update_layout(height=600, yaxis_range=[0, 110])
fig.show()

In [ ]:
# Cell A: Find images with most misclassifications (pretrained models only)

worst_images_query = """
WITH image_performance AS (
    SELECT 
        p.image_id,
        i.filename,
        i.file_path,
        COUNT(*) as total_predictions,
        SUM(CASE WHEN p.predicted_value = gt.value THEN 0 ELSE 1 END) as errors,
        ROUND(
            100.0 * SUM(CASE WHEN p.predicted_value = gt.value THEN 1 ELSE 0 END) / COUNT(*),
            1
        ) as accuracy_percentage
    FROM predictions p
    JOIN images i ON p.image_id = i.image_id
    JOIN ground_truth_labels gt ON p.image_id = gt.image_id AND p.label_name = gt.label_name
    JOIN analysis_runs ar ON p.analysis_run_id = ar.analysis_run_id
    WHERE ar.analysis_type = 'llm_classification' or ar.analysis_type = 'yolo_classification'  -- Pretrained models only
    GROUP BY p.image_id, i.filename, i.file_path
)
SELECT 
    image_id,
    filename,
    file_path,
    total_predictions,
    errors,
    accuracy_percentage
FROM image_performance
ORDER BY errors DESC, accuracy_percentage ASC
LIMIT 10
"""

worst_images = pd.read_sql(worst_images_query, conn)
print("🔴 Top 3 Worst-Performing Images:\n")
display(worst_images)

In [ ]:
# Cell B: Display the worst-performing images (flexible grid)

from PIL import Image
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import math

# PARAMETER: Change this to show more/fewer images
NUM_IMAGES_TO_SHOW = 10  # TODO: Change this number (e.g., 6, 9, 12)

# Get the images to display
images_to_display = worst_images.head(NUM_IMAGES_TO_SHOW)

if len(images_to_display) > 0:
    # Calculate grid dimensions (3 images per row)
    images_per_row = 3
    num_rows = math.ceil(len(images_to_display) / images_per_row)
    num_cols = min(len(images_to_display), images_per_row)
    
    # Create subplot titles
    subplot_titles = [
        f"{row['filename']}<br>Accuracy: {row['accuracy_percentage']:.1f}%<br>Errors: {row['errors']}/{row['total_predictions']}"
        for _, row in images_to_display.iterrows()
    ]
    
    # Create subplots
    fig = make_subplots(
        rows=num_rows,
        cols=images_per_row,
        horizontal_spacing=0.15,
        vertical_spacing=0.2,
        subplot_titles=subplot_titles
    )
    
    # Add images
    for idx, (_, row) in enumerate(images_to_display.iterrows()):
        # Calculate row and column position (1-indexed)
        plot_row = (idx // images_per_row) + 1
        plot_col = (idx % images_per_row) + 1
        
        try:
            img = Image.open(row['file_path'])
            fig.add_trace(
                go.Image(z=img),
                row=plot_row, col=plot_col
            )
        except Exception as e:
            print(f"Could not load image: {row['file_path']} - {e}")
    
    # Style adjustments
    for annotation in fig.layout.annotations:
        annotation.font.size = 10
    
    fig.update_xaxes(showticklabels=False)
    fig.update_yaxes(showticklabels=False)
    fig.update_layout(
        title_text=f"Top {NUM_IMAGES_TO_SHOW} Most Frequently Misclassified Images",
        height=400 * num_rows,  # Scale height with number of rows
        width=1200,
        showlegend=False
    )
    
    fig.show()
else:
    print("No classification data found")

In [ ]:
worst_images_query = """

SELECT 
    a.image_id,
    a.filename,
    file_path,
    b.label_name,
    b.value,
    c.label_name,
    c.predicted_value
FROM images a
join ground_truth_labels b
on a.image_id = b.image_id
join predictions c
on b.image_id = c.image_id and b.label_name = c.label_name
where a.image_id = 67
"""

worst_images = pd.read_sql(worst_images_query, conn)
print("🔴 Top 3 Worst-Performing Images:\n")
display(worst_images)

---
# Additional Explorations

Add your own cells below to explore other aspects of the database.

**Tips:**
- Copy any cell above and modify the query
- Change parameters (analysis_run_id, model_name, label_name)
- Use different Plotly chart types: `px.scatter()`, `px.line()`, `px.box()`, etc.
- Combine multiple queries with SQL JOINs or pandas merges

---

In [ ]:
# Example: Your custom exploration here
# Copy cells from above and modify as needed

---
# Cleanup
---

In [ ]:
# Close database connection
conn.close()
print("✅ Database connection closed")